In [1]:
import os, sys, time, math
print(os.cpu_count())
print(sys.version)


8
3.13.7 (main, Sep 18 2025, 19:47:49) [Clang 20.1.4 ]


In [2]:

# Quick exploration: check existing workspace files & spec
import os
for root, dirs, files in os.walk('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481'):
 for f in files:
 p = os.path.join(root, f)
 print(p, os.path.getsize(p))
 # avoid descending too deep


/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/-PROMPT-v6-DATASET.md 25848
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.kernel_llm_logs_1.txt 0
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/memory/ead8dad7-2c8b-48d6-bfcb-911508844481_memory_heron_c0_20260530T182649217126.jsonl 37265
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.prompts/BOOTSTRAP.md 3909
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.prompts/MEMORY.md 319
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.prompts/HEARTBEAT.md 1997
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.config/ipython/profile_default/ipython_kernel_config.py 671
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.config/ipython/profile_default/heron_matplotlib_theme.py 10305
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.config/ipython/profile_default/__pycache__/heron_matplotlib_theme.cpython-313.pyc 13242
/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/.config/ipython/profile_default/__pycache__/ipython_kernel_config.cpython-313.pyc 

In [3]:

# Standard Davenport-Heilbronn function:
# Defined modulo 5 via a character-like function f(n) with f(1)=1, f(2)=tan(theta), f(3)=-tan(theta), f(4)=-1, f(5)=0 (period 5),
# where tan(theta) = (sqrt(10 - 2 sqrt(5)) - 2)/(sqrt(5) - 1) chosen so the functional equation holds.
# Equivalently L_DH(s) = 5^{-s} * (zeta(s,1/5) + tan(theta) zeta(s,2/5) - tan(theta) zeta(s,3/5) - zeta(s,4/5))
# Functional equation:
# xi(s) = (pi/5)^{-s/2} * Gamma((s+1)/2) * L_DH(s)
# xi(s) = xi(1-s)
# Hardy-Z type function on the critical line:
# theta_DH(t) = arg of (pi/5)^{-(1/2+it)/2} Gamma((1/2+it+1)/2)/2 ... actually
# Z(t) = (pi/5)^{-it/2} * Gamma((1/2+it+1)/2) / |Gamma((1/2+it+1)/2)| * (pi/5)^{1/4} ... 
# Let me use a simpler real-valued formulation:
# chi(s) = (pi/5)^{1/2 - s} * Gamma((1-s+1)/2) / Gamma((s+1)/2)
# Then xi(s) = (pi/5)^{-s/2} Gamma((s+1)/2) L_DH(s) and using xi real on critical line:
# Actually a cleaner approach: define Z(t) = phi(t) * L_DH(1/2+it) where phi(t) is unimodular so Z is real.
# phi(t) = (pi/5)^{it/2} * Gamma((1/2-it+1)/2) / |Gamma((1/2+it+1)/2)| but that doesn't quite work either.
# The standard construction: from xi(s)=xi(1-s) with xi(s)=(pi/5)^{-s/2} Gamma((s+1)/2) L_DH(s),
# define Z(t) = (pi/5)^{-(1/2+it)/2} * Gamma((1/2+it+1)/2) * L_DH(1/2+it) * e^{-i*Theta(t)}
# where Theta(t) = arg[(pi/5)^{-(1/2+it)/2} Gamma((3/4 + it/2))] ... so Z(t) is real.
# Equivalently: Z(t) = exp(i*theta(t)) * L_DH(1/2 + it), where
# theta(t) = arg[ Gamma(3/4 + it/2) * (pi/5)^{-it/2} ] (using xi rewrite)
# Let's verify this gives real values.

import mpmath as mp
mp.mp.dps = 40

# tan(theta) constant for DH function
sqrt5 = mp.sqrt(5)
tan_theta = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
print("tan_theta =", tan_theta)

def L_DH(s):
 s = mp.mpc(s)
 return mp.power(5, -s) * (
 mp.zeta(s, mp.mpf(1)/5)
 + tan_theta * mp.zeta(s, mp.mpf(2)/5)
 - tan_theta * mp.zeta(s, mp.mpf(3)/5)
 - mp.zeta(s, mp.mpf(4)/5)
 )

# Check known DH zeros (off-critical-line):
for sigma, t in [(0.808517,85.699348),(0.650786,114.163343),(0.574355,166.479306),(0.724258,176.702461)]:
 v = L_DH(mp.mpc(sigma, t))
 print(f"L_DH({sigma},{t}) = {complex(v):.3e}, |.|={abs(v):.3e}")


tan_theta = 0.2840790438404122960282918323931261690911


TypeError: unsupported format string passed to mpf.__format__

In [4]:

for sigma, t in [(0.808517,85.699348),(0.650786,114.163343),(0.574355,166.479306),(0.724258,176.702461)]:
 v = L_DH(mp.mpc(sigma, t))
 print(f"L_DH({sigma},{t}) = {complex(v)}, |.|={float(abs(v)):.3e}")


L_DH(0.808517,85.699348) = (-3.420835057223351e-07-5.540608371175485e-07j), |.|=6.512e-07
L_DH(0.650786,114.163343) = (-3.833486276101339e-05-1.2539720058854122e-05j), |.|=4.033e-05
L_DH(0.574355,166.479306) = (-6.324608350633461e-07+2.53974756907477e-07j), |.|=6.815e-07
L_DH(0.724258,176.702461) = (2.8314163492453745e-07-3.7608615302266734e-07j), |.|=4.708e-07


In [5]:

# Good - matches the dataset description: first, third, fourth ~ 10^-7, second ~ 4e-5 (transcription artifact).
# Now define Hardy-Z function. Functional equation:
# xi(s) = (pi/5)^{-s/2} Gamma((s+1)/2) L_DH(s), xi(s) = xi(1-s).
# On the critical line s = 1/2 + it:
# xi(1/2+it) = (pi/5)^{-(1/2+it)/2} Gamma(3/4 + it/2) L_DH(1/2 + it)
# Since xi(1/2+it) = xi(1/2-it) = conj(xi(1/2+it)) (xi real on critical line by functional equation + reality of coefficients)
# Wait — DH function has real coefficients? f(1)=1, f(2)=tan_theta, f(3)=-tan_theta, f(4)=-1, all real. 
# So L_DH(conj(s)) = conj(L_DH(s)), and xi inherits real on critical line.
# Therefore Z(t) = xi(1/2+it) / |Gamma(3/4+it/2)| * (pi/5)^{1/4} would be real, but simpler:
# Define theta_DH(t) = arg[(pi/5)^{-it/2} * Gamma(3/4 + it/2)]
# Then Z(t) = exp(i*theta_DH(t)) * L_DH(1/2 + it) is real.

def theta_DH(t):
 t = mp.mpf(t)
 # arg of (pi/5)^{-it/2} = -t/2 * log(pi/5)
 a = -t/2 * mp.log(mp.pi/5)
 g = mp.loggamma(mp.mpc(mp.mpf(3)/4, t/2))
 return a + g.imag

def Z_DH(t):
 t = mp.mpf(t)
 th = theta_DH(t)
 val = mp.exp(mp.mpc(0, th)) * L_DH(mp.mpc(mp.mpf(1)/2, t))
 return val.real # imag should be ~0

# Sanity: compute imag part at a few values
import time
t0 = time.time()
for tt in [10.0, 20.0, 50.0, 85.7, 100.0]:
 val = mp.exp(mp.mpc(0, theta_DH(tt))) * L_DH(mp.mpc(mp.mpf(1)/2, tt))
 print(f"t={tt}: Re={float(val.real):.6e}, Im={float(val.imag):.3e}")
print("elapsed:", time.time()-t0)


t=10.0: Re=1.919392e+00, Im=-9.440e-41
t=20.0: Re=1.958042e+00, Im=-4.384e-40
t=50.0: Re=-6.477924e-01, Im=3.183e-40
t=85.7: Re=-3.568319e-01, Im=5.174e-40
t=100.0: Re=8.706124e-01, Im=8.801e-40
elapsed: 0.08117413520812988


In [6]:

# Imag is essentially 0 (machine eps at dps=40). Hardy-Z is real and well-defined.
# DH function is known to have ~3 zeros below t~14 and approximately N(T) ~ (T/(2*pi)) * log(T/(2*pi*e/5)) ... 
# Practical: Riemann-von Mangoldt for DH: N(T) ~ T/(2*pi) * log(5*T/(2*pi)) - T/(2*pi) (approx)
# Estimate T for N=5000:
import scipy.optimize as opt

def N_DH_estimate(T):
 return T/(2*math.pi) * math.log(5*T/(2*math.pi)) - T/(2*math.pi)

# find T such that N_DH(T) ~ 5000
for T in [3000, 3500, 4000, 4500, 5000]:
 print(T, N_DH_estimate(T))


3000 3236.2224328781563
3500 3861.4612877220434
4000 4498.107339316029
4500 5144.726642247403
5000 5800.206170198717


In [7]:

# So T ~ 4500 gives ~5145 zeros - consistent with the project budget. Time-cost feasibility check:
# At dps=50 they reported ~1.5 s/zero. With 8 cores ideally 0.2 s/zero -> 5000 zeros ~ 1000 s. OK.
# Let me benchmark Z evaluation at dps=50.
mp.mp.dps = 50

def L_DH_dps(s, _tan=tan_theta):
 return mp.power(5, -s) * (
 mp.zeta(s, mp.mpf(1)/5)
 + _tan * mp.zeta(s, mp.mpf(2)/5)
 - _tan * mp.zeta(s, mp.mpf(3)/5)
 - mp.zeta(s, mp.mpf(4)/5)
 )

def Z_DH_dps(t):
 t = mp.mpf(t)
 a = -t/2 * mp.log(mp.pi/5)
 g_im = mp.loggamma(mp.mpc(mp.mpf(3)/4, t/2)).imag
 th = a + g_im
 val = mp.exp(mp.mpc(0, th)) * L_DH_dps(mp.mpc(mp.mpf(1)/2, t))
 return val.real

import time
t0 = time.time()
for tt in [10.0, 100.0, 1000.0, 4500.0]:
 v = Z_DH_dps(tt)
 print(f"t={tt}: Z={float(v):.6e}")
print("elapsed:", time.time()-t0, "per eval:", (time.time()-t0)/4)


t=10.0: Z=1.919392e+00
t=100.0: Z=8.706124e-01
t=1000.0: Z=8.630124e-01


t=4500.0: Z=1.612968e+00
elapsed: 0.6465742588043213 per eval: 0.1616438627243042


In [8]:

# ~0.16 s/eval at dps=50 around large t. With T=4500 and step ~0.5, that's 9000 evals + bisections.
# Probably ~20 evals per sign-change (initial scan + refine). 
# Total serial: 9000*0.16 + 5000*15*0.16 = 1440 + 12000 = ~3-4 hours serial. With 8 cores ~25-30 min.
# 
# Strategy: Step size needs to be small enough to catch all zeros. DH zero spacing ~ 2*pi/log(5*T/(2*pi))
# At T=4500: spacing ~ 2*pi/log(5*4500/(2*pi)) = 2*pi/log(3580) ~ 6.28/8.18 ~ 0.77
# Use step=0.25 to be safe.

T_low = 0.0
T_high = 4600.0 # a bit over to ensure >=5000 zeros, then trim
step = mp.mpf('0.25')
print(f"Number of grid points: {int((T_high-T_low)/float(step))}")
print(f"Estimated zeros in [0,{T_high}]: {N_DH_estimate(T_high):.0f}")


Number of grid points: 18400
Estimated zeros in [0,4600.0]: 5275


In [9]:

# 18400 evals just for scan. At 0.16 s/eval => 2944 s serial. Add ~5000 zeros * 10-15 evals for refinement => ~12000 s. Too long.
# Use a coarser scan (step=0.5) and refine. Average spacing decreases monotonically; at low t spacing larger, so adaptive.
# Better: use coarse step=0.5, then for each bracket where Z changes sign, find zero via mpmath findroot or brentq.
# To catch close zeros (twins), step=0.5 is risky. Let's check spacing at t=4500: ~0.77. step=0.5 should be OK.
# But near t=0, spacing is larger. So 0.5 OK.

# Bigger concern: avg refine evals. brentq with full dps=50 precision needs ~50 iterations? log2(0.5/1e-50) ~ 168 bisections.
# Better: use mpmath findroot with secant once we have a bracket — typically ~5-8 iters.
# Use scipy.optimize.brentq on a float-evaluated wrapper for fast bracket narrowing, then mpmath findroot for high precision polish.

# Actually simpler: use mpmath.findroot with secant in the bracket; should converge quadratically to dps=50 in ~6-8 iters.
# Total: 5000 * 8 * 0.16 = 6400 s + scan 2944 s = ~9300 s serial. Parallel 8x => ~1200 s. Manageable.

# Let me first write a worker that handles a contiguous block. Then test on small block serially to estimate.

# Block worker design:
# Input: (t_lo, t_hi, step, dps)
# Output: list of zero ordinates as strings (for portability via multiprocessing)

# Test serial on a small range first.
mp.mp.dps = 50

def find_zeros_in_block(t_lo, t_hi, step=0.5, dps=50, tan_theta_str=None):
 import mpmath as mp
 mp.mp.dps = dps
 if tan_theta_str is None:
 sqrt5 = mp.sqrt(5)
 tt = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
 else:
 tt = mp.mpf(tan_theta_str)
 log_pi_5 = mp.log(mp.pi / 5)

 def Z(t):
 t = mp.mpf(t)
 a = -t/2 * log_pi_5
 g_im = mp.loggamma(mp.mpc(mp.mpf(3)/4, t/2)).imag
 th = a + g_im
 s = mp.mpc(mp.mpf(1)/2, t)
 L = mp.power(5, -s) * (
 mp.zeta(s, mp.mpf(1)/5)
 + tt * mp.zeta(s, mp.mpf(2)/5)
 - tt * mp.zeta(s, mp.mpf(3)/5)
 - mp.zeta(s, mp.mpf(4)/5)
 )
 return (mp.exp(mp.mpc(0, th)) * L).real

 zeros = []
 t = mp.mpf(t_lo)
 z_prev = Z(t)
 step = mp.mpf(step)
 t_hi = mp.mpf(t_hi)
 while t < t_hi:
 t_next = t + step
 if t_next > t_hi:
 t_next = t_hi
 z_next = Z(t_next)
 if z_prev == 0:
 zeros.append(t)
 elif (z_prev > 0) != (z_next > 0):
 # bracket - refine using mpmath findroot
 try:
 r = mp.findroot(Z, (t, t_next), solver='anderson', tol=mp.mpf(10)**(-dps+2))
 except Exception:
 # fallback to bisection-like refine
 a_, b_ = t, t_next
 za_ = z_prev
 for _ in range(200):
 m_ = (a_+b_)/2
 zm_ = Z(m_)
 if zm_ == 0 or (b_-a_) < mp.mpf(10)**(-dps+2):
 break
 if (za_>0) != (zm_>0):
 b_ = m_
 else:
 a_ = m_
 za_ = zm_
 r = (a_+b_)/2
 zeros.append(r)
 t = t_next
 z_prev = z_next
 return [mp.nstr(z, dps) for z in zeros]

# small test
t0 = time.time()
zlist = find_zeros_in_block(0.0, 50.0, step=0.5, dps=50)
print("found:", len(zlist), "in", time.time()-t0, "s")
for z in zlist[:5]:
 print(z)


found: 21 in 4.71059250831604 s
5.0941598445710949256987955170797974750670744531091
8.9399144080965232447176128773475666256121616213856
12.133545425800742403702547753765281383126636124036
14.404003112277498207270097673803851118378453181957
17.130239400572489600773954527308750679976252956357


In [10]:

# Looks reasonable - DH zeros in [0,50]. Known: first DH zero ~ 5.094.
# 21 zeros in 50 in 4.7 s → about 0.22 s per zero serial. Scaled to 5275 zeros over [0,4600] => ~1170 s serial.
# (Actually it's a bit slower at high t due to expensive zeta(s, p/5) for large t.) Let me benchmark middle range.

t0 = time.time()
zlist2 = find_zeros_in_block(1000.0, 1050.0, step=0.5, dps=50)
print("found:", len(zlist2), "in", time.time()-t0, "s")

t0 = time.time()
zlist3 = find_zeros_in_block(4500.0, 4550.0, step=0.5, dps=50)
print("found:", len(zlist3), "in", time.time()-t0, "s")


found: 45 in 73.29414415359497 s


found: 60 in 296.3582932949066 s


In [11]:

# Ouch - at t=4500 it's ~5 s/zero! Because mp.zeta(s, p/5) for large Im(s) is slow.
# Total serial estimate: ~300 s per 50-unit block at high t. We have ~92 blocks of 50 units total. 
# A trapezoidal estimate: avg cost increases roughly with log(T)*something. At t=25: 0.22 s/z; t=1025: 1.6 s/z; t=4525: 5 s/z.
# Rough total serial: integrate cost*density over t. Approximate:
# zeros density ~ (1/2pi) * log(5t/(2pi)). cost per zero ~ c(t).
# Just estimate: avg ~ 2 s/zero over 5000 zeros => 10000 s serial. With 8 cores => 1250 s wall (~21 min). Feasible.

# But the per-eval Z time grows. Let me check: at t=4500, Z eval is ~0.16 s (from earlier test). With ~10 evals/zero, that's 1.6 s.
# But here we see ~5 s/zero. The scan is happening at step=0.5, and 60 zeros in 50 units means 100 scan points + 60 brackets w/ ~8 refines each = 100+480 = 580 evals → 580 * 0.16 = 93 s ... but actual was 296 s.
# So actual eval time at t~4500 is ~0.5 s, not 0.16. Probably anderson solver does many internal evaluations.

# Let me try faster solver: secant with mpmath.findroot, providing both endpoints. Or bisection.
# Bisection: 168 iterations * 0.16 s = 27 s per zero - much worse.
# Better strategy: use scipy.optimize.brentq on a numpy-float version of Z for crude bracket (fast), then mpmath polish.

# Actually let me check: maybe the bottleneck is many internal evaluations by anderson.
# Try with secant solver explicitly:

mp.mp.dps = 50
sqrt5 = mp.sqrt(5)
tt_global = (mp.sqrt(10 - 2*sqrt5) - 2) / (sqrt5 - 1)
log_pi_5 = mp.log(mp.pi / 5)

def Z_dps50(t):
 t = mp.mpf(t)
 a = -t/2 * log_pi_5
 g_im = mp.loggamma(mp.mpc(mp.mpf(3)/4, t/2)).imag
 th = a + g_im
 s = mp.mpc(mp.mpf(1)/2, t)
 L = mp.power(5, -s) * (
 mp.zeta(s, mp.mpf(1)/5)
 + tt_global * mp.zeta(s, mp.mpf(2)/5)
 - tt_global * mp.zeta(s, mp.mpf(3)/5)
 - mp.zeta(s, mp.mpf(4)/5)
 )
 return (mp.exp(mp.mpc(0, th)) * L).real

# Time a single eval at t=4500:
t0 = time.time()
for _ in range(5):
 Z_dps50(4500.0 + _*0.1)
print("eval at t=4500:", (time.time()-t0)/5, "s")

t0 = time.time()
for _ in range(5):
 Z_dps50(1000.0 + _*0.1)
print("eval at t=1000:", (time.time()-t0)/5, "s")


eval at t=4500: 0.5225594520568848 s


eval at t=1000: 0.1016477108001709 s


In [12]:

# So eval at t=4500: 0.52 s/eval; t=1000: 0.10 s/eval. Hurwitz zeta is expensive at high t.
# Scan budget: 18400 evals (step=0.5 over [0,4600]) with average ~ 0.3 s = 5520 s serial.
# Refine: 5000 zeros * 8 evals * 0.3 s = 12000 s serial.
# Total serial ~ 17500 s. With 8 cores ~2200 s wall = ~37 minutes. Feasible within 60 min budget.

# Let me also test whether brentq on a float Z would work for rough bracket, then mpmath polish. 
# But float precision can't even capture L_DH at high t — Im(zeta(s,1/5)) for t=4500 is enormous, 
# need high precision. Stick with mpmath.

# Better idea: use mp.dps=25 for the scan (much faster), then refine at dps=50.
# Let me test:
def Z_dps_var(t, dps=25):
 old = mp.mp.dps
 mp.mp.dps = dps
 try:
 sqrt5l = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5l) - 2) / (sqrt5l - 1)
 lp5 = mp.log(mp.pi/5)
 t = mp.mpf(t)
 a = -t/2 * lp5
 g_im = mp.loggamma(mp.mpc(mp.mpf(3)/4, t/2)).imag
 th = a + g_im
 s = mp.mpc(mp.mpf(1)/2, t)
 L = mp.power(5, -s) * (
 mp.zeta(s, mp.mpf(1)/5)
 + ttl * mp.zeta(s, mp.mpf(2)/5)
 - ttl * mp.zeta(s, mp.mpf(3)/5)
 - mp.zeta(s, mp.mpf(4)/5)
 )
 val = (mp.exp(mp.mpc(0, th)) * L).real
 return val
 finally:
 mp.mp.dps = old

t0 = time.time()
for _ in range(5):
 Z_dps_var(4500.0 + _*0.1, dps=25)
print("eval at t=4500 dps=25:", (time.time()-t0)/5, "s")

t0 = time.time()
for _ in range(5):
 Z_dps_var(4500.0 + _*0.1, dps=30)
print("eval at t=4500 dps=30:", (time.time()-t0)/5, "s")


eval at t=4500 dps=25: 0.2360368251800537 s


eval at t=4500 dps=30: 0.27389183044433596 s


In [13]:

# dps=25 ~ 0.24 s vs 0.52 s at dps=50. ~2x speedup. Use dps=30 for scan, dps=50 for polish.
# Let me redesign the worker to use the two-phase approach:

def find_zeros_in_block_v2(t_lo, t_hi, step=0.5, dps_scan=30, dps_final=50):
 import mpmath as mp
 
 def make_Z(dps):
 # closure capturing precision
 mp.mp.dps = dps
 sqrt5l = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5l) - 2) / (sqrt5l - 1)
 lp5 = mp.log(mp.pi/5)
 one_fifth = mp.mpf(1)/5
 two_fifth = mp.mpf(2)/5
 three_fifth = mp.mpf(3)/5
 four_fifth = mp.mpf(4)/5
 three_quarter = mp.mpf(3)/4
 half = mp.mpf(1)/2
 def Z(t):
 t = mp.mpf(t)
 a = -t/2 * lp5
 g_im = mp.loggamma(mp.mpc(three_quarter, t/2)).imag
 th = a + g_im
 s = mp.mpc(half, t)
 L = mp.power(5, -s) * (
 mp.zeta(s, one_fifth)
 + ttl * mp.zeta(s, two_fifth)
 - ttl * mp.zeta(s, three_fifth)
 - mp.zeta(s, four_fifth)
 )
 return (mp.exp(mp.mpc(0, th)) * L).real
 return Z
 
 # SCAN at dps_scan
 mp.mp.dps = dps_scan
 Z_scan = make_Z(dps_scan)
 
 brackets = []
 t = mp.mpf(t_lo)
 step_mp = mp.mpf(step)
 t_hi_mp = mp.mpf(t_hi)
 z_prev = Z_scan(t)
 while t < t_hi_mp:
 t_next = t + step_mp
 if t_next > t_hi_mp:
 t_next = t_hi_mp
 z_next = Z_scan(t_next)
 if (z_prev > 0) != (z_next > 0) and z_prev != 0 and z_next != 0:
 brackets.append((t, t_next, z_prev, z_next))
 elif z_prev == 0:
 brackets.append((t, t, mp.mpf(0), mp.mpf(0)))
 t = t_next
 z_prev = z_next
 
 # POLISH at dps_final using brentq-style bisection then secant
 mp.mp.dps = dps_final
 Z_pol = make_Z(dps_final)
 zeros = []
 tol = mp.mpf(10)**(-dps_final+3)
 for a, b, za, zb in brackets:
 if a == b:
 zeros.append(a)
 continue
 # Use mp.findroot with secant — start from midpoint
 try:
 r = mp.findroot(Z_pol, (a, b), solver='anderson', tol=tol)
 except Exception:
 # fallback bisection
 za = Z_pol(a); zb = Z_pol(b)
 lo, hi = a, b
 for _ in range(200):
 m = (lo+hi)/2
 zm = Z_pol(m)
 if zm == 0 or (hi-lo) < tol:
 break
 if (za > 0) != (zm > 0):
 hi = m
 else:
 lo = m
 za = zm
 r = (lo+hi)/2
 zeros.append(r)
 return [mp.nstr(z, dps_final) for z in zeros]

# Test on a moderate-cost block
t0 = time.time()
zlist = find_zeros_in_block_v2(1000.0, 1050.0, step=0.5, dps_scan=30, dps_final=50)
print(f"found: {len(zlist)} zeros in {time.time()-t0:.1f}s at t~1000")
print(zlist[0], "...", zlist[-1])

t0 = time.time()
zlist = find_zeros_in_block_v2(4500.0, 4550.0, step=0.5, dps_scan=30, dps_final=50)
print(f"found: {len(zlist)} zeros in {time.time()-t0:.1f}s at t~4500")


found: 45 zeros in 71.5s at t~1000
1000.5653832066416780775487613015659144243268439222 ... 1049.4512885529874463117564672378877749060910474263


found: 60 zeros in 271.4s at t~4500


In [14]:

# Mild speedup at high t. Roughly 25% reduction. Let me try a different solver - 'secant' starting from midpoint:
# but we'd need a single starting point. Actually findroot with secant takes a tuple of 2 starting points.

# Alternative: do bracket refinement in float using mp dps=15 first (very fast), then polish at dps=50.
# Hurwitz at dps=15 might still be slow at t=4500. Test:

t0 = time.time()
for _ in range(5):
 Z_dps_var(4500.0 + _*0.1, dps=20)
print("eval at t=4500 dps=20:", (time.time()-t0)/5, "s")

t0 = time.time()
for _ in range(5):
 Z_dps_var(4500.0 + _*0.1, dps=15)
print("eval at t=4500 dps=15:", (time.time()-t0)/5, "s")


eval at t=4500 dps=20: 0.33690638542175294 s


eval at t=4500 dps=15: 0.2670483112335205 s


In [15]:

# The expense at high t is fundamental (Hurwitz zeta convergence at large imaginary part). dps=15 only 25% faster than dps=30.
# Let's just commit and parallelize. 
# Total cost estimate: roughly serial 50-unit block takes ~70s at t~1000, ~270s at t~4500. 
# Integrate: assume linear ramp from 5s/block (at t=0) to 270s/block (at t=4500). 
# Avg ~ 137 s/block * 92 blocks = ~12600 s serial. With 8 cores ~1600 s wall. ~27 minutes.
# 
# Time budget: 3600 s total - already used ~10 min. Have ~50 min left. Doable but tight.
# 
# To play it safe, let me ALSO consider just running serial and showing the parallelization works on a subset:
# Actually no, the task says to compute N=5000 zeros. Let's just go for it with parallel execution.

# Plan: chunks of 100-unit blocks. 46 chunks. 8 workers. Use multiprocessing.Pool.
# Each worker: find_zeros_in_block_v2 with scan dps=30, final dps=50.

# IMPORTANT: workers need to be picklable. Define top-level function in a temp .py module file.

# Write worker as standalone module:
from e14c.filesystem import write_file

worker_code = '''
import mpmath as mp

def find_zeros_in_block_v2(args):
 t_lo, t_hi, step, dps_scan, dps_final = args
 
 def make_Z(dps):
 mp.mp.dps = dps
 sqrt5l = mp.sqrt(5)
 ttl = (mp.sqrt(10 - 2*sqrt5l) - 2) / (sqrt5l - 1)
 lp5 = mp.log(mp.pi/5)
 one_fifth = mp.mpf(1)/5
 two_fifth = mp.mpf(2)/5
 three_fifth = mp.mpf(3)/5
 four_fifth = mp.mpf(4)/5
 three_quarter = mp.mpf(3)/4
 half = mp.mpf(1)/2
 def Z(t):
 t = mp.mpf(t)
 a = -t/2 * lp5
 g_im = mp.loggamma(mp.mpc(three_quarter, t/2)).imag
 th = a + g_im
 s = mp.mpc(half, t)
 L = mp.power(5, -s) * (
 mp.zeta(s, one_fifth)
 + ttl * mp.zeta(s, two_fifth)
 - ttl * mp.zeta(s, three_fifth)
 - mp.zeta(s, four_fifth)
 )
 return (mp.exp(mp.mpc(0, th)) * L).real
 return Z
 
 # Scan
 mp.mp.dps = dps_scan
 Z_scan = make_Z(dps_scan)
 
 brackets = []
 t = mp.mpf(t_lo)
 step_mp = mp.mpf(step)
 t_hi_mp = mp.mpf(t_hi)
 z_prev = Z_scan(t)
 while t < t_hi_mp:
 t_next = t + step_mp
 if t_next > t_hi_mp:
 t_next = t_hi_mp
 z_next = Z_scan(t_next)
 if z_prev != 0 and z_next != 0 and (z_prev > 0) != (z_next > 0):
 brackets.append((float(t), float(t_next)))
 elif z_prev == 0:
 brackets.append((float(t), float(t)))
 t = t_next
 z_prev = z_next
 
 # Polish
 mp.mp.dps = dps_final
 Z_pol = make_Z(dps_final)
 zeros = []
 tol = mp.mpf(10)**(-dps_final+3)
 for a_f, b_f in brackets:
 if a_f == b_f:
 zeros.append(mp.mpf(a_f))
 continue
 a = mp.mpf(a_f); b = mp.mpf(b_f)
 try:
 r = mp.findroot(Z_pol, (a, b), solver='anderson', tol=tol)
 except Exception:
 za = Z_pol(a); zb = Z_pol(b)
 lo, hi = a, b
 for _ in range(250):
 m = (lo+hi)/2
 zm = Z_pol(m)
 if zm == 0 or (hi-lo) < tol:
 break
 if (za > 0) != (zm > 0):
 hi = m
 else:
 lo = m
 za = zm
 r = (lo+hi)/2
 zeros.append(r)
 return [mp.nstr(z, dps_final, strip_zeros=False) for z in zeros]
'''

write_file('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_worker.py', worker_code)
print("worker written")


worker written


In [16]:

# Quick smoke test of worker module from disk:
import sys
sys.path.insert(0, '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481')
import importlib
import ldh_worker
importlib.reload(ldh_worker)

import time
t0 = time.time()
out = ldh_worker.find_zeros_in_block_v2((0.0, 50.0, 0.5, 30, 50))
print(f"found {len(out)} zeros in {time.time()-t0:.2f}s")
print(out[0])
print(out[-1])


found 21 zeros in 4.23s
5.0941598445710949256987955170797974750670744531091
48.477786580349021973481636534205266034099274636945


In [17]:

# Good. Now run in parallel across multiple cores. We have 8 CPUs.
# Block size 100 units => 46 blocks. With 8 workers, well-load-balanced.
# Use chunksize=1 so big blocks get distributed dynamically.

import multiprocessing as mp_mod
import time

t_lo_global = 0.0
t_hi_global = 4600.0
block_size = 100.0
step = 0.5
dps_scan = 30
dps_final = 50

# Build block args, with reversed order so big blocks (high t) go first - better load balancing
blocks = []
t = t_lo_global
while t < t_hi_global:
 t_end = min(t + block_size, t_hi_global)
 blocks.append((t, t_end, step, dps_scan, dps_final))
 t = t_end

# Reverse to start from high-t blocks (slow) first
blocks_reversed = list(reversed(blocks))
print(f"Number of blocks: {len(blocks)} of size {block_size}")
print(f"First few (reversed): {blocks_reversed[:3]}")


Number of blocks: 46 of size 100.0
First few (reversed): [(4500.0, 4600.0, 0.5, 30, 50), (4400.0, 4500.0, 0.5, 30, 50), (4300.0, 4400.0, 0.5, 30, 50)]


In [18]:

# Estimate before running: ~70s/block at t~1000, ~300s/block at t~4500, ~10s/block at t~100.
# Crude integral: at t (mid block), per-zero cost ~ a + b*t. Empirically: 
# block(0-50): 4s for 21 zeros = 0.2s/z
# block(1000-1050): 70s/45 zeros = 1.55 s/z 
# block(4500-4550): 270s/60 zeros = 4.5 s/z
# Per-zero cost looks linear in log(t)-ish. Let's just run it parallel.

# Estimate: total work ≈ sum over blocks of (~zeros_in_block * cost(t))
# Quick estimate: integrate (zeros density * cost). 
# zeros_density(t) ~ (1/2pi) log(5t/2pi)
# cost(t) ~ 0.5 + 0.001*t (rough fit)
# Total work = ∫₀^4600 density(t) * cost(t) dt
import numpy as np
def density(t):
 if t < 1: return 0
 return (1/(2*math.pi)) * math.log(5*t/(2*math.pi))
def cost_per_zero(t):
 # Interpolate: 0.2 at t=25, 1.55 at t=1025, 4.5 at t=4525
 return 0.2 + 0.0009 * t

T_arr = np.linspace(1, 4600, 4600)
work_serial = sum(density(t) * cost_per_zero(t) for t in T_arr)
print(f"Estimated serial work: {work_serial:.0f} s = {work_serial/60:.1f} min")
print(f"Parallel (8 cores ideal): {work_serial/8/60:.1f} min")


Estimated serial work: 12735 s = 212.3 min
Parallel (8 cores ideal): 26.5 min


In [19]:

# OK 26 min ideal. With overhead and imbalance, expect 30-35 min. We've used ~12 min, have 48 min. Tight but doable.
# Let me start the run with imap_unordered, save partial results as they come in.

import multiprocessing as mp_mod
import time, pickle, os, json

result_dir = '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481'
partial_file = os.path.join(result_dir, 'ldh_partial.pkl')

# Make sure ldh_worker available in spawned processes:
# multiprocessing on Linux uses fork by default - inherits sys.path. Good.

n_workers = 8
results = {} # block_idx -> list of zero strings

t0 = time.time()
# Use spawn explicitly? Fork should be fine on Linux. Use 'fork' for env inheritance.
ctx = mp_mod.get_context('fork')
print(f"Starting pool with {n_workers} workers at {time.strftime('%H:%M:%S')}")

with ctx.Pool(processes=n_workers) as pool:
 # Map blocks_reversed; track which index corresponds
 block_idx_map = {i: blocks_reversed[i] for i in range(len(blocks_reversed))}
 futures = []
 for i, b in enumerate(blocks_reversed):
 futures.append((i, pool.apply_async(ldh_worker.find_zeros_in_block_v2, (b,))))
 
 completed = 0
 for i, fut in futures:
 try:
 res = fut.get()
 results[i] = res
 completed += 1
 elapsed = time.time() - t0
 block = blocks_reversed[i]
 print(f"[{elapsed:6.1f}s] block {i+1}/{len(blocks)} done: t in [{block[0]:.0f},{block[1]:.0f}], "
 f"{len(res)} zeros, total so far: {sum(len(v) for v in results.values())}")
 except Exception as e:
 print(f"block {i} FAILED: {e}")
 results[i] = []

print(f"\nTotal elapsed: {time.time()-t0:.1f}s")
print(f"Total zeros found: {sum(len(v) for v in results.values())}")

# Save partial
with open(partial_file, 'wb') as f:
 pickle.dump({'results': results, 'blocks_reversed': blocks_reversed}, f)
print("Saved partial to", partial_file)


TimeoutError: Code execution timed out after 900.0 seconds

In [20]:

# 15-min cell timeout. Need shorter cells. Use Python script run in background + poll.
# Strategy: launch as a background process via subprocess, polling for completion.
# 
# Write a self-contained driver script that:
# - imports ldh_worker
# - runs parallel
# - saves results incrementally to a json/pkl file
# Then launch via subprocess.Popen and poll.

driver_code = '''
import sys, os, time, pickle, json
sys.path.insert(0, "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481")
import multiprocessing as mp_mod
import ldh_worker

t_lo_global = 0.0
t_hi_global = 4600.0
block_size = 100.0
step = 0.5
dps_scan = 30
dps_final = 50

blocks = []
t = t_lo_global
while t < t_hi_global:
 t_end = min(t + block_size, t_hi_global)
 blocks.append((t, t_end, step, dps_scan, dps_final))
 t = t_end
blocks_reversed = list(reversed(blocks))

n_workers = 8
results = {}
t0 = time.time()

partial_file = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl"
log_file = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log"

def log(msg):
 with open(log_file, "a") as f:
 f.write(f"[{time.time()-t0:7.1f}s] {msg}\\n")
 f.flush()

# Clear log
open(log_file, "w").close()
log(f"Starting {len(blocks)} blocks with {n_workers} workers")

ctx = mp_mod.get_context("fork")
with ctx.Pool(processes=n_workers) as pool:
 futures = []
 for i, b in enumerate(blocks_reversed):
 futures.append((i, pool.apply_async(ldh_worker.find_zeros_in_block_v2, (b,))))
 
 completed = 0
 for i, fut in futures:
 try:
 res = fut.get()
 results[i] = res
 completed += 1
 block = blocks_reversed[i]
 total_so_far = sum(len(v) for v in results.values())
 log(f"block {i+1}/{len(blocks)} done [{block[0]:.0f},{block[1]:.0f}]: "
 f"{len(res)} zeros, completed={completed}, total_zeros={total_so_far}")
 # Save partial periodically
 if completed % 5 == 0 or completed == len(blocks):
 with open(partial_file, "wb") as f:
 pickle.dump({"results": results, "blocks_reversed": blocks_reversed}, f)
 log(f"saved partial after {completed} blocks")
 except Exception as e:
 log(f"block {i} FAILED: {e}")
 results[i] = []

with open(partial_file, "wb") as f:
 pickle.dump({"results": results, "blocks_reversed": blocks_reversed}, f)

log(f"DONE. Total elapsed: {time.time()-t0:.1f}s, total zeros: {sum(len(v) for v in results.values())}")
'''

from e14c.filesystem import write_file
write_file('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py', driver_code)
print("driver written")


driver written


In [21]:

# Launch as background subprocess (nohup-style)
import subprocess, os
log_path = '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log'
stdout_path = '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_stdout.log'

# Clear stale files
for p in [log_path, stdout_path]:
 if os.path.exists(p):
 os.remove(p)

proc = subprocess.Popen(
 ['python', '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py'],
 stdout=open(stdout_path, 'w'),
 stderr=subprocess.STDOUT,
 cwd='/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481',
)
print("PID:", proc.pid)

# Save PID for monitoring
with open('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_pid.txt', 'w') as f:
 f.write(str(proc.pid))


PID: 138


In [22]:

import time, os
# Poll progress
time.sleep(30)
if os.path.exists(log_path):
 print(open(log_path).read())
print("---stdout---")
if os.path.exists(stdout_path):
 print(open(stdout_path).read()[-2000:])


[ 0.0s] Starting 46 blocks with 8 workers

---stdout---



In [23]:

import time
time.sleep(120)
print(open(log_path).read())


[ 0.0s] Starting 46 blocks with 8 workers



In [24]:

# No blocks completed yet after 2.5 minutes. The first blocks (highest t) take ~300s each.
# That's normal. Wait more. Check if process still running.
import subprocess
res = subprocess.run(['ps', '-p', '138', '-o', 'pid,stat,etime,cmd'], capture_output=True, text=True)
print(res.stdout)
print(res.stderr)
# Also check children:
res = subprocess.run(['ps', '-ef'], capture_output=True, text=True)
print([l for l in res.stdout.split('\n') if 'python' in l][:20])


PermissionError: [Errno 13] Permission denied: 'ps'

In [25]:

# Check /proc/138 directly
import os
print("Process exists:", os.path.exists('/proc/138'))
if os.path.exists('/proc/138/status'):
 with open('/proc/138/status') as f:
 for line in f.readlines()[:10]:
 print(line.rstrip())
# children
import glob
print("\nPython processes:")
for d in glob.glob('/proc/[0-9]*'):
 try:
 with open(os.path.join(d, 'cmdline')) as f:
 cmd = f.read().replace('\x00', ' ')
 if 'python' in cmd or 'run_parallel' in cmd:
 print(d, cmd[:100])
 except:
 pass


Process exists: True
Name:	python
State:	S (sleeping)
Tgid:	138
Pid:	138
PPid:	20
TracerPid:	0
Uid:	1500	1500	1500	1500
Gid:	1500	1500	1500	1500
FDSize:	512
Groups:	1500 1400

Python processes:
/proc/1 python -m uvicorn sandbox_service.app:app --host 0.0.0.0 --port 8080 
/proc/17 su -l kerneluser -s /bin/bash -c umask 002 && export NEW_RELIC_DISTRIBUTED_TRACING_ENABLED=true && e
/proc/20 /app/kernel_env/bin/python -Xfrozen_modules=off -m ipykernel_launcher -f /workspace/.kernel_meta_ead
/proc/138 python /workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py 
/proc/139 python /workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py 
/proc/140 python /workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py 
/proc/141 python /workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py 
/proc/142 python /workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py 
/proc/143 python /workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel.py 
/proc/144 python

In [26]:

# Good — 8 workers + 1 parent. All running. CPU info via /proc:
def get_cpu_pct(pid):
 try:
 with open(f'/proc/{pid}/stat') as f:
 parts = f.read().split()
 utime = int(parts[13])
 stime = int(parts[14])
 starttime = int(parts[21])
 # uptime
 with open('/proc/uptime') as f:
 uptime = float(f.read().split()[0])
 clock_ticks = 100
 total_time = utime + stime
 seconds = uptime - (starttime / clock_ticks)
 cpu = 100 * (total_time / clock_ticks) / seconds if seconds > 0 else 0
 return cpu, total_time/clock_ticks, seconds
 except:
 return None
for pid in [138,139,140,141,142,143,144,145,146]:
 print(pid, get_cpu_pct(pid))


138 (0.12915724895059755, 0.24, 185.8199999999997)
139 (94.88284406140606, 176.15, 185.64999999999964)
140 (94.86102133160986, 176.1, 185.63999999999942)
141 (94.84997037116887, 176.07, 185.6299999999992)
142 (94.8278648779701, 176.01, 185.60999999999967)
143 (94.8706896551727, 176.08, 185.59999999999945)
144 (94.92968371140725, 176.18, 185.58999999999924)
145 (94.8755253798901, 176.07, 185.57999999999993)
146 (94.9989221815049, 176.28, 185.5599999999995)


In [27]:

# Perfect — all 8 workers running at ~95% CPU. Need to wait. Sleep more.
import time
time.sleep(600)
print(open(log_path).read())


[ 0.0s] Starting 46 blocks with 8 workers



In [28]:

# Critical warning - 833 s remaining. Need to wrap up. Let me check progress carefully.
import time, os
print(open(log_path).read())
print("Wall clock since start (approx since starting subprocess):")
print("--- partial file size ---")
if os.path.exists('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl'):
 print(os.path.getsize('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl'))
else:
 print("partial not yet saved")


[ 0.0s] Starting 46 blocks with 8 workers

Wall clock since start (approx since starting subprocess):
--- partial file size ---
partial not yet saved


In [29]:

# No blocks completed yet after ~13 minutes. The high-t blocks were submitted FIRST (reversed), 
# each taking 300-500+s. We won't get a single one done before timeout.
# This was a mistake in load balancing — I should have submitted in NATURAL order so low-t blocks finish first
# providing many partial results, OR mixed ordering.
# 
# CRITICAL: I need to (a) kill the current run, (b) launch a NEW run that processes LOW-T blocks first
# so we get SOMETHING, and (c) restrict scope to demonstrate feasibility but get less than 5000 zeros.
#
# Time budget remaining ~13 min. Can't get 5000 zeros. Honest reporting: feasibility-demonstrated but not completed.
#
# Best move: 
# 1. Kill current process.
# 2. Launch a quick run that does low-t blocks first, output as many zeros as possible.
# 3. Aggregate partial results, perform R2 validation, save what we have, report honestly.

import os, signal

# Kill all worker processes 
for pid in [138,139,140,141,142,143,144,145,146]:
 try:
 os.kill(pid, signal.SIGTERM)
 print(f"killed {pid}")
 except Exception as e:
 print(f"could not kill {pid}: {e}")


killed 138
killed 139
killed 140
killed 141
killed 142
killed 143
killed 144
killed 145
killed 146


In [30]:

# Launch fresh run with natural ordering (low t first) AND smaller blocks for finer-grained results.
# Reduce dps to 30 only (skip 50 polish) just to demonstrate the parallel pipeline works end-to-end.
# Actually, keep dps=50 for correctness but use small blocks (25 units) so we get results fast.

driver_code2 = '''
import sys, os, time, pickle, json
sys.path.insert(0, "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481")
import multiprocessing as mp_mod
import ldh_worker

t_lo_global = 0.0
t_hi_global = 4600.0
block_size = 25.0
step = 0.5
dps_scan = 30
dps_final = 50

blocks = []
t = t_lo_global
while t < t_hi_global:
 t_end = min(t + block_size, t_hi_global)
 blocks.append((t, t_end, step, dps_scan, dps_final))
 t = t_end
# Natural order - low t first so we get many quick results
n_workers = 8
results = {}
t0 = time.time()

partial_file = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl"
log_file = "/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log"

def log(msg):
 with open(log_file, "a") as f:
 f.write(f"[{time.time()-t0:7.1f}s] {msg}\\n")
 f.flush()

open(log_file, "w").close()
log(f"Starting {len(blocks)} blocks of size {block_size} with {n_workers} workers")

ctx = mp_mod.get_context("fork")
with ctx.Pool(processes=n_workers) as pool:
 futures = []
 for i, b in enumerate(blocks):
 futures.append((i, pool.apply_async(ldh_worker.find_zeros_in_block_v2, (b,))))
 completed = 0
 for i, fut in futures:
 try:
 res = fut.get()
 results[i] = res
 completed += 1
 block = blocks[i]
 total_so_far = sum(len(v) for v in results.values())
 log(f"block {i+1}/{len(blocks)} [{block[0]:.0f},{block[1]:.0f}]: "
 f"{len(res)} z, completed={completed}, tot={total_so_far}")
 if completed % 3 == 0 or completed == len(blocks):
 with open(partial_file, "wb") as f:
 pickle.dump({"results": results, "blocks": blocks}, f)
 except Exception as e:
 log(f"block {i} FAILED: {e}")
 results[i] = []

with open(partial_file, "wb") as f:
 pickle.dump({"results": results, "blocks": blocks}, f)
log(f"DONE elapsed={time.time()-t0:.1f}s total={sum(len(v) for v in results.values())}")
'''

from e14c.filesystem import write_file
write_file('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel2.py', driver_code2)

import subprocess, time
# Remove old files
for p in ['/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_partial.pkl',
 '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log']:
 if os.path.exists(p): os.remove(p)

proc = subprocess.Popen(
 ['python', '/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/run_parallel2.py'],
 stdout=open('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_stdout.log', 'w'),
 stderr=subprocess.STDOUT,
)
print("PID:", proc.pid)
time.sleep(15)
print(open('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log').read())


PID: 151


[ 0.0s] Starting 184 blocks of size 25.0 with 8 workers
[ 3.0s] block 1/184 [0,25]: 8 z, completed=1, tot=8
[ 4.7s] block 2/184 [25,50]: 13 z, completed=2, tot=21
[ 6.5s] block 3/184 [50,75]: 16 z, completed=3, tot=37
[ 7.2s] block 4/184 [75,100]: 15 z, completed=4, tot=52
[ 8.3s] block 5/184 [100,125]: 16 z, completed=5, tot=68



In [31]:

# Excellent! Now we're getting blocks done quickly at low t. Let me wait ~10 minutes for as much as possible.
import time
time.sleep(420) # 7 min
print(open('/workspace/ead8dad7-2c8b-48d6-bfcb-911508844481/ldh_run.log').read()[-3000:])


TimeoutError: Code execution timed out after 169.0 seconds